In [4]:
# =========================================================
# REALISTIC HUMAN-IN-THE-LOOP (HITL) WORKFLOW
# AI Insurance Claim Review System
# USING LANGGRAPH (NO AGENTS)
# =========================================================

# Install
!pip install -q langgraph

from typing import TypedDict
from langgraph.graph import StateGraph, END


# =========================================================
# STATE
# =========================================================

class InsuranceState(TypedDict):
    customer_name: str
    claim_amount: int
    claim_reason: str

    risk_level: str
    ai_decision: str

    human_decision: str
    final_status: str


# =========================================================
# NODE 1 — ANALYZE CLAIM RISK
# =========================================================

def analyze_claim(state: InsuranceState):

    amount = state["claim_amount"]
    reason = state["claim_reason"].lower()

    # Simple business rules
    if amount > 100000:
        risk = "HIGH"

    elif "accident" in reason:
        risk = "MEDIUM"

    else:
        risk = "LOW"

    print("\n==============================")
    print("CLAIM RISK ANALYSIS")
    print("==============================")
    print(f"Claim Amount : ₹{amount}")
    print(f"Detected Risk: {risk}")

    return {
        "risk_level": risk
    }


# =========================================================
# NODE 2 — AI DECISION
# =========================================================

def ai_review(state: InsuranceState):

    risk = state["risk_level"]
    amount = state["claim_amount"]

    if risk == "LOW":
        decision = "AUTO APPROVE"

    elif risk == "MEDIUM":
        decision = "REQUIRES HUMAN REVIEW"

    else:
        decision = "ESCALATE TO SENIOR MANAGER"

    print("\n==============================")
    print("AI CLAIM REVIEW")
    print("==============================")
    print(f"AI Decision: {decision}")

    return {
        "ai_decision": decision
    }


# =========================================================
# NODE 3 — HUMAN APPROVAL (HITL)
# =========================================================

def human_review(state: InsuranceState):

    print("\n==============================")
    print("HUMAN CLAIM OFFICER REVIEW")
    print("==============================")

    print(f"Customer      : {state['customer_name']}")
    print(f"Claim Amount  : ₹{state['claim_amount']}")
    print(f"Claim Reason  : {state['claim_reason']}")
    print(f"Risk Level    : {state['risk_level']}")
    print(f"AI Suggestion : {state['ai_decision']}")

    print("\nChoose Decision:")
    print("1 → APPROVE")
    print("2 → REJECT")
    print("3 → NEED MORE DOCUMENTS")

    choice = input("\nEnter decision: ")

    mapping = {
        "1": "APPROVED",
        "2": "REJECTED",
        "3": "PENDING DOCUMENTS"
    }

    decision = mapping.get(choice, "INVALID DECISION")

    return {
        "human_decision": decision
    }


# =========================================================
# NODE 4 — FINALIZE CLAIM
# =========================================================

def finalize_claim(state: InsuranceState):

    final = state["human_decision"]

    print("\n==============================")
    print("FINAL CLAIM STATUS")
    print("==============================")

    print(f"Customer : {state['customer_name']}")
    print(f"Final Status : {final}")

    return {
        "final_status": final
    }


# =========================================================
# BUILD GRAPH
# =========================================================

builder = StateGraph(InsuranceState)

builder.add_node("analyze_claim", analyze_claim)
builder.add_node("ai_review", ai_review)
builder.add_node("human_review", human_review)
builder.add_node("finalize_claim", finalize_claim)

builder.set_entry_point("analyze_claim")

builder.add_edge("analyze_claim", "ai_review")
builder.add_edge("ai_review", "human_review")
builder.add_edge("human_review", "finalize_claim")
builder.add_edge("finalize_claim", END)

graph = builder.compile()


# =========================================================
# RUN WORKFLOW
# =========================================================

result = graph.invoke({

    "customer_name": "John",

    "claim_amount": 150000,

    "claim_reason": "Major vehicle accident on highway"

})


# =========================================================
# FINAL RESULT OBJECT
# =========================================================

print("\n==============================")
print("COMPLETE STATE")
print("==============================")

print(result)


CLAIM RISK ANALYSIS
Claim Amount : ₹150000
Detected Risk: HIGH

AI CLAIM REVIEW
AI Decision: ESCALATE TO SENIOR MANAGER

HUMAN CLAIM OFFICER REVIEW
Customer      : John
Claim Amount  : ₹150000
Claim Reason  : Major vehicle accident on highway
Risk Level    : HIGH
AI Suggestion : ESCALATE TO SENIOR MANAGER

Choose Decision:
1 → APPROVE
2 → REJECT
3 → NEED MORE DOCUMENTS

Enter decision: 1

FINAL CLAIM STATUS
Customer : John
Final Status : APPROVED

COMPLETE STATE
{'customer_name': 'John', 'claim_amount': 150000, 'claim_reason': 'Major vehicle accident on highway', 'risk_level': 'HIGH', 'ai_decision': 'ESCALATE TO SENIOR MANAGER', 'human_decision': 'APPROVED', 'final_status': 'APPROVED'}


In [5]:
# =========================================================
# LANGGRAPH CHECKPOINTING DEMO
# REALISTIC EXAMPLE:
# AI LOAN APPLICATION PROCESSING SYSTEM
# =========================================================

# Install
!pip install -q langgraph

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


# =========================================================
# STATE
# =========================================================

class LoanState(TypedDict):

    customer_name: str
    salary: int
    credit_score: int
    loan_amount: int

    eligibility: str
    risk: str
    final_decision: str


# =========================================================
# NODE 1 — VERIFY ELIGIBILITY
# =========================================================

def verify_eligibility(state: LoanState):

    salary = state["salary"]

    print("\n==============================")
    print("STEP 1: VERIFY ELIGIBILITY")
    print("==============================")

    if salary >= 50000:
        eligibility = "ELIGIBLE"
    else:
        eligibility = "NOT ELIGIBLE"

    print(f"Salary: ₹{salary}")
    print(f"Eligibility: {eligibility}")

    return {
        "eligibility": eligibility
    }


# =========================================================
# NODE 2 — RISK ANALYSIS
# =========================================================

def risk_analysis(state: LoanState):

    credit = state["credit_score"]

    print("\n==============================")
    print("STEP 2: RISK ANALYSIS")
    print("==============================")

    if credit >= 750:
        risk = "LOW RISK"

    elif credit >= 600:
        risk = "MEDIUM RISK"

    else:
        risk = "HIGH RISK"

    print(f"Credit Score: {credit}")
    print(f"Risk Level : {risk}")

    return {
        "risk": risk
    }


# =========================================================
# NODE 3 — FINAL DECISION
# =========================================================

def final_decision(state: LoanState):

    print("\n==============================")
    print("STEP 3: FINAL LOAN DECISION")
    print("==============================")

    if (
        state["eligibility"] == "ELIGIBLE"
        and state["risk"] != "HIGH RISK"
    ):

        decision = "LOAN APPROVED"

    else:
        decision = "LOAN REJECTED"

    print(f"Final Decision: {decision}")

    return {
        "final_decision": decision
    }


# =========================================================
# BUILD GRAPH
# =========================================================

builder = StateGraph(LoanState)

builder.add_node("verify_eligibility", verify_eligibility)
builder.add_node("risk_analysis", risk_analysis)
builder.add_node("final_decision", final_decision)

builder.set_entry_point("verify_eligibility")

builder.add_edge("verify_eligibility", "risk_analysis")
builder.add_edge("risk_analysis", "final_decision")
builder.add_edge("final_decision", END)


# =========================================================
# CHECKPOINTER
# =========================================================

# Stores workflow progress
memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)


# =========================================================
# CONFIG (THREAD ID)
# =========================================================

config = {
    "configurable": {
        "thread_id": "loan-001"
    }
}


# =========================================================
# RUN WORKFLOW
# =========================================================

result = graph.invoke(

    {

        "customer_name": "Ahmed",

        "salary": 80000,

        "credit_score": 720,

        "loan_amount": 500000

    },

    config=config
)

print("\n==============================")
print("WORKFLOW FINISHED")
print("==============================")


# =========================================================
# VIEW SAVED CHECKPOINT STATE
# =========================================================

saved_state = graph.get_state(config)

print("\n==============================")
print("CHECKPOINTED STATE")
print("==============================")

print(saved_state.values)



STEP 1: VERIFY ELIGIBILITY
Salary: ₹80000
Eligibility: ELIGIBLE

STEP 2: RISK ANALYSIS
Credit Score: 720
Risk Level : MEDIUM RISK

STEP 3: FINAL LOAN DECISION
Final Decision: LOAN APPROVED

WORKFLOW FINISHED

CHECKPOINTED STATE
{'customer_name': 'Ahmed', 'salary': 80000, 'credit_score': 720, 'loan_amount': 500000, 'eligibility': 'ELIGIBLE', 'risk': 'MEDIUM RISK', 'final_decision': 'LOAN APPROVED'}


In [6]:
# =========================================================
# LANGGRAPH CHECKPOINTING DEMO
# RESUME WORKFLOW FROM ANY STEP
#
# REALISTIC EXAMPLE:
# BANK LOAN PROCESSING SYSTEM
# =========================================================

# Install
!pip install -q langgraph

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


# =========================================================
# STATE
# =========================================================

class LoanState(TypedDict):

    customer_name: str
    salary: int
    credit_score: int
    loan_amount: int

    eligibility: str
    risk_level: str
    manager_approval: str
    final_status: str


# =========================================================
# NODE 1 — ELIGIBILITY CHECK
# =========================================================

def eligibility_check(state: LoanState):

    print("\n==============================")
    print("STEP 1 — ELIGIBILITY CHECK")
    print("==============================")

    if state["salary"] >= 50000:
        eligibility = "ELIGIBLE"
    else:
        eligibility = "NOT ELIGIBLE"

    print(f"Customer : {state['customer_name']}")
    print(f"Eligibility : {eligibility}")

    return {
        "eligibility": eligibility
    }


# =========================================================
# NODE 2 — RISK ANALYSIS
# =========================================================

def risk_analysis(state: LoanState):

    print("\n==============================")
    print("STEP 2 — RISK ANALYSIS")
    print("==============================")

    score = state["credit_score"]

    if score >= 750:
        risk = "LOW RISK"

    elif score >= 600:
        risk = "MEDIUM RISK"

    else:
        risk = "HIGH RISK"

    print(f"Credit Score : {score}")
    print(f"Risk Level   : {risk}")

    return {
        "risk_level": risk
    }


# =========================================================
# NODE 3 — MANAGER APPROVAL
# =========================================================

def manager_review(state: LoanState):

    print("\n==============================")
    print("STEP 3 — MANAGER REVIEW")
    print("==============================")

    print(f"Loan Amount : ₹{state['loan_amount']}")
    print(f"Risk Level  : {state['risk_level']}")

    approval = input(
        "\nApprove Loan? (yes/no): "
    )

    return {
        "manager_approval": approval
    }


# =========================================================
# NODE 4 — FINAL DECISION
# =========================================================

def final_decision(state: LoanState):

    print("\n==============================")
    print("STEP 4 — FINAL DECISION")
    print("==============================")

    if (
        state["eligibility"] == "ELIGIBLE"
        and state["manager_approval"].lower() == "yes"
    ):

        status = "LOAN APPROVED"

    else:
        status = "LOAN REJECTED"

    print(f"Final Status : {status}")

    return {
        "final_status": status
    }


# =========================================================
# BUILD GRAPH
# =========================================================

builder = StateGraph(LoanState)

builder.add_node("eligibility_check", eligibility_check)
builder.add_node("risk_analysis", risk_analysis)
builder.add_node("manager_review", manager_review)
builder.add_node("final_decision", final_decision)

builder.set_entry_point("eligibility_check")

builder.add_edge("eligibility_check", "risk_analysis")
builder.add_edge("risk_analysis", "manager_review")
builder.add_edge("manager_review", "final_decision")
builder.add_edge("final_decision", END)


# =========================================================
# ENABLE CHECKPOINTING
# =========================================================

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)


# =========================================================
# THREAD CONFIG
# =========================================================

config = {
    "configurable": {
        "thread_id": "loan-process-001"
    }
}


# =========================================================
# INITIAL INPUT
# =========================================================

initial_state = {

    "customer_name": "Ahmed",

    "salary": 85000,

    "credit_score": 720,

    "loan_amount": 400000

}


# =========================================================
# RUN WORKFLOW STEP BY STEP
# =========================================================

print("\nSTARTING WORKFLOW...\n")

for event in graph.stream(

    initial_state,

    config=config

):

    print("\nEVENT:")
    print(event)

    # Simulate stopping system anytime
    stop = input(
        "\nStop workflow here? (yes/no): "
    )

    if stop.lower() == "yes":

        print("\nWORKFLOW STOPPED")
        print("Checkpoint Saved Automatically")

        break


# =========================================================
# LATER... RESUME WORKFLOW
# =========================================================

resume = input(
    "\nDo you want to resume workflow later? (yes/no): "
)

if resume.lower() == "yes":

    print("\nRESUMING FROM LAST CHECKPOINT...\n")

    for event in graph.stream(

        None,   # IMPORTANT
        config=config

    ):

        print("\nRESUMED EVENT:")
        print(event)


# =========================================================
# VIEW FINAL SAVED STATE
# =========================================================

saved_state = graph.get_state(config)

print("\n==============================")
print("FINAL SAVED STATE")
print("==============================")

print(saved_state.values)


STARTING WORKFLOW...


STEP 1 — ELIGIBILITY CHECK
Customer : Ahmed
Eligibility : ELIGIBLE

EVENT:
{'eligibility_check': {'eligibility': 'ELIGIBLE'}}

Stop workflow here? (yes/no): yes

WORKFLOW STOPPED
Checkpoint Saved Automatically

Do you want to resume workflow later? (yes/no): yes

RESUMING FROM LAST CHECKPOINT...


STEP 2 — RISK ANALYSIS
Credit Score : 720
Risk Level   : MEDIUM RISK

RESUMED EVENT:
{'risk_analysis': {'risk_level': 'MEDIUM RISK'}}

STEP 3 — MANAGER REVIEW
Loan Amount : ₹400000
Risk Level  : MEDIUM RISK

Approve Loan? (yes/no): yes

RESUMED EVENT:
{'manager_review': {'manager_approval': 'yes'}}

STEP 4 — FINAL DECISION
Final Status : LOAN APPROVED

RESUMED EVENT:
{'final_decision': {'final_status': 'LOAN APPROVED'}}

FINAL SAVED STATE
{'customer_name': 'Ahmed', 'salary': 85000, 'credit_score': 720, 'loan_amount': 400000, 'eligibility': 'ELIGIBLE', 'risk_level': 'MEDIUM RISK', 'manager_approval': 'yes', 'final_status': 'LOAN APPROVED'}


In [7]:
# =========================================================
# LANGGRAPH CHECKPOINTING DEMO
# SAVE + LIST + RESUME CHECKPOINTS
#
# REALISTIC ENTERPRISE WORKFLOW
# =========================================================

# Install
!pip install -q langgraph

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
import uuid


# =========================================================
# STATE
# =========================================================

class LoanState(TypedDict):

    customer_name: str
    salary: int
    credit_score: int
    loan_amount: int

    eligibility: str
    risk_level: str
    manager_approval: str
    final_status: str


# =========================================================
# NODE 1 — ELIGIBILITY CHECK
# =========================================================

def eligibility_check(state: LoanState):

    print("\n==============================")
    print("STEP 1 — ELIGIBILITY CHECK")
    print("==============================")

    if state["salary"] >= 50000:
        eligibility = "ELIGIBLE"
    else:
        eligibility = "NOT ELIGIBLE"

    print(f"Customer : {state['customer_name']}")
    print(f"Eligibility : {eligibility}")

    return {
        "eligibility": eligibility
    }


# =========================================================
# NODE 2 — RISK ANALYSIS
# =========================================================

def risk_analysis(state: LoanState):

    print("\n==============================")
    print("STEP 2 — RISK ANALYSIS")
    print("==============================")

    score = state["credit_score"]

    if score >= 750:
        risk = "LOW RISK"

    elif score >= 600:
        risk = "MEDIUM RISK"

    else:
        risk = "HIGH RISK"

    print(f"Credit Score : {score}")
    print(f"Risk Level   : {risk}")

    return {
        "risk_level": risk
    }


# =========================================================
# NODE 3 — MANAGER REVIEW
# =========================================================

def manager_review(state: LoanState):

    print("\n==============================")
    print("STEP 3 — MANAGER REVIEW")
    print("==============================")

    print(f"Loan Amount : ₹{state['loan_amount']}")
    print(f"Risk Level  : {state['risk_level']}")

    approval = input(
        "\nApprove Loan? (yes/no): "
    )

    return {
        "manager_approval": approval
    }


# =========================================================
# NODE 4 — FINAL DECISION
# =========================================================

def final_decision(state: LoanState):

    print("\n==============================")
    print("STEP 4 — FINAL DECISION")
    print("==============================")

    if (
        state["eligibility"] == "ELIGIBLE"
        and state["manager_approval"].lower() == "yes"
    ):

        status = "LOAN APPROVED"

    else:
        status = "LOAN REJECTED"

    print(f"Final Status : {status}")

    return {
        "final_status": status
    }


# =========================================================
# BUILD GRAPH
# =========================================================

builder = StateGraph(LoanState)

builder.add_node("eligibility_check", eligibility_check)
builder.add_node("risk_analysis", risk_analysis)
builder.add_node("manager_review", manager_review)
builder.add_node("final_decision", final_decision)

builder.set_entry_point("eligibility_check")

builder.add_edge("eligibility_check", "risk_analysis")
builder.add_edge("risk_analysis", "manager_review")
builder.add_edge("manager_review", "final_decision")
builder.add_edge("final_decision", END)


# =========================================================
# CHECKPOINTER
# =========================================================

memory = MemorySaver()

graph = builder.compile(
    checkpointer=memory
)


# =========================================================
# CREATE UNIQUE THREAD ID
# =========================================================

thread_id = str(uuid.uuid4())

config = {
    "configurable": {
        "thread_id": thread_id
    }
}

print("\nTHREAD ID:", thread_id)


# =========================================================
# INITIAL INPUT
# =========================================================

initial_state = {

    "customer_name": "Ahmed",

    "salary": 85000,

    "credit_score": 710,

    "loan_amount": 500000

}


# =========================================================
# RUN WORKFLOW
# =========================================================

print("\nSTARTING WORKFLOW...\n")

for event in graph.stream(

    initial_state,

    config=config

):

    print("\nEVENT:")
    print(event)

    # Ask whether to save checkpoint
    save = input(
        "\nSave checkpoint here? (yes/no): "
    )

    if save.lower() == "yes":

        print("\nCHECKPOINT SAVED")

        stop = input(
            "\nStop workflow now? (yes/no): "
        )

        if stop.lower() == "yes":

            print("\nWORKFLOW STOPPED")
            break


# =========================================================
# SHOW CHECKPOINT DETAILS
# =========================================================

print("\n==============================")
print("AVAILABLE CHECKPOINTS")
print("==============================")

checkpoints = list(
    graph.get_state_history(config)
)

for idx, checkpoint in enumerate(checkpoints):

    print(f"\nCheckpoint #{idx}")

    print("Checkpoint ID:")
    print(checkpoint.config)

    print("\nSaved State:")
    print(checkpoint.values)

    print("\nNext Node:")
    print(checkpoint.next)


# =========================================================
# RESUME FROM CHECKPOINT
# =========================================================

resume = input(
    "\nDo you want to resume workflow? (yes/no): "
)

if resume.lower() == "yes":

    print("\nAvailable Checkpoints:")

    for idx, checkpoint in enumerate(checkpoints):
        print(f"{idx} → {checkpoint.next}")

    selected = int(
        input("\nEnter checkpoint number: ")
    )

    selected_checkpoint = checkpoints[selected]

    print("\n==============================")
    print("RESUMING FROM CHECKPOINT")
    print("==============================")

    for event in graph.stream(

        None,

        config=selected_checkpoint.config

    ):

        print("\nRESUMED EVENT:")
        print(event)


# =========================================================
# FINAL STATE
# =========================================================

final_state = graph.get_state(config)

print("\n==============================")
print("FINAL SAVED STATE")
print("==============================")

print(final_state.values)


THREAD ID: 3a3ea64e-d874-481f-8323-a480ce3936bc

STARTING WORKFLOW...


STEP 1 — ELIGIBILITY CHECK
Customer : Ahmed
Eligibility : ELIGIBLE

EVENT:
{'eligibility_check': {'eligibility': 'ELIGIBLE'}}

Save checkpoint here? (yes/no): yes

CHECKPOINT SAVED

Stop workflow now? (yes/no): yes

WORKFLOW STOPPED

AVAILABLE CHECKPOINTS

Checkpoint #0
Checkpoint ID:
{'configurable': {'thread_id': '3a3ea64e-d874-481f-8323-a480ce3936bc', 'checkpoint_ns': '', 'checkpoint_id': '1f14f750-c86e-64d4-8001-673b14b33f3b'}}

Saved State:
{'customer_name': 'Ahmed', 'salary': 85000, 'credit_score': 710, 'loan_amount': 500000, 'eligibility': 'ELIGIBLE'}

Next Node:
('risk_analysis',)

Checkpoint #1
Checkpoint ID:
{'configurable': {'thread_id': '3a3ea64e-d874-481f-8323-a480ce3936bc', 'checkpoint_ns': '', 'checkpoint_id': '1f14f750-c86b-637d-8000-f12cb24df824'}}

Saved State:
{'customer_name': 'Ahmed', 'salary': 85000, 'credit_score': 710, 'loan_amount': 500000}

Next Node:
('eligibility_check',)

Checkpoint #2